# 06 -- Visualization

`qwenmoe/viz/` holds `common.py::capture_trace`, which sources real numbers
from a real checkpoint via `QwenMoEModel.forward_with_trace`. This used to
also be the foundation every Manim scene in this project built its
animations on, but **all Manim scene code (and the `manim` dependency) has
since been removed from the project** -- `capture_trace` is now consumed
by `token_journey/`'s web walkthrough instead.

*Part of the [training-my-moe code tour](00_overview.ipynb) -- see also: [config](01_config.ipynb) - [tokenizer](02_tokenizer.ipynb) - [data](03_data.ipynb) - [model](04_model.ipynb) - [train](05_train.ipynb) - [viz](06_viz.ipynb).*

In [1]:
from pathlib import Path
import os

# Walk up until we find pyproject.toml, so these cells work no matter where Jupyter's cwd starts.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
os.chdir(repo_root)
print("Working directory:", Path.cwd())

Working directory: /Users/sjoseph/Documents/Study/training-my-moe


## `capture_trace` (`common.py`) -- the one place every visualization gets real numbers from
Loads a checkpoint, encodes a prompt with the exact tokenizer it was
trained with, and runs `forward_with_trace` -- never illustrative or
fabricated numbers.

In [2]:
from qwenmoe.viz.common import capture_trace
from qwenmoe.tokenizer.base import build_tokenizer_from_meta
from qwenmoe.train.trainer import load_checkpoint

checkpoint_path = "runs/smoke_test/checkpoints/step_1000.pt"
bundle = capture_trace(checkpoint_path, "ROMEO:")
tokenizer = build_tokenizer_from_meta(load_checkpoint(checkpoint_path)["tokenizer_meta"])

print("token_ids:", bundle.token_ids, "->", repr(tokenizer.decode(bundle.token_ids)))
print("embeddings shape:", tuple(bundle.embeddings.shape))
print("logits shape:", tuple(bundle.logits.shape), " probs shape:", tuple(bundle.probs.shape))

top_id = bundle.probs.argmax().item()
print(f"top predicted next character: {tokenizer.decode([top_id])!r}  (p={bundle.probs[top_id].item():.4f})")

token_ids: [30, 27, 25, 17, 27, 10] -> 'ROMEO:'
embeddings shape: (6, 64)
logits shape: (6, 65)  probs shape: (65,)
top predicted next character: '\n'  (p=0.8828)


### One block's `LayerTrace` in detail

In [3]:
layer0 = bundle.layers[0]
has_pos_trace = layer0.rope_angles_or_alibi_bias is not None
print("rope_angles_or_alibi_bias shape:", tuple(layer0.rope_angles_or_alibi_bias.shape) if has_pos_trace else None)
print("attn_scores_post_softmax shape:", tuple(layer0.attn_scores_post_softmax.shape))
print("router_probs shape:", tuple(layer0.router_probs.shape))
print("topk_indices:")
print(layer0.topk_indices)
print("expert_outputs present for experts:", list(layer0.expert_outputs.keys()),
      "of", layer0.router_probs.shape[-1], "total routed experts")

rope_angles_or_alibi_bias shape: (6, 16)
attn_scores_post_softmax shape: (4, 6, 6)
router_probs shape: (6, 4)
topk_indices:
tensor([[0, 3],
        [3, 0],
        [3, 1],
        [3, 1],
        [3, 2],
        [3, 1]])
expert_outputs present for experts: [0, 1, 2, 3] of 4 total routed experts


## Manim scenes -- removed

This project previously had 5 formal Manim scenes (task 13: embedding+RoPE,
attention, MoE routing, RMSNorm+SwiGLU, output projection) and 2
training-observability scenes (task 14: loss curve, expert utilization),
plus 9 supplementary "preview" scenes and a `scripts/render_viz.py`
rendering CLI. **All of this Manim scene code, its test, and the `manim`
dependency have since been deleted from the project entirely.** Real-number
visualization now lives in `architecture_diagram.html`, `token_journey/`,
`ablation_study/`, and this repo's Plotly-based notebooks instead.